# Caso G · 04 Agentes especialistas de calidad (mock)

> _Tutorial · Caso de uso: **G — Calidad con agentes** · Capa Medallion: **transversal** · Spec: `docs/specs/synthetic-bms/02-domain-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Implementar 3 agentes-mock que actúan como evaluadores: validador de plata, auditor de MLflow, evaluador de chatbot.


## 2. Qué se aprende

- Patrón `@tool` / `@function_tool`.
- Cómo combinar herramientas en un agente.
- Cómo mockear LLM si no hay clave.


## 3. Contexto del caso de uso

Curso evalúa la calidad del proyecto con un agente que llama a herramientas. Usaremos un cliente fake.


## 4. Relación con CENTINELA+

Los agentes pueden correr como tarea diaria desde el Dashboard Adapter.


## 5. Relación con Medallion

Transversal.


## 6. Datos de entrada

Mocks.


## 7. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [1]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


ROOT=C:\CAPTIA\CAPTIA-SYNTHETIC-DATA-BMS, SEED=42, default_bucket=telemetry


## 8. Schema CAPTIA esperado

No aplica.


## 9. Carga de datos o mock

Definimos 3 tools.


In [2]:
def validate_silver_layer(asset_id: str = "AULA01") -> dict:
    return {
        "asset_id": asset_id, "tags_present": True, "completeness_pct": 99.2,
        "outliers": 0, "verdict": "OK",
    }

def audit_mlflow_experiment(name: str = "case_B_baseline_2026") -> dict:
    return {
        "experiment": name, "n_runs": 4, "best_MAE": 12.4, "lakefs_tag_referenced": True,
        "verdict": "OK",
    }

def evaluate_chatbot_response(question: str, expected: str) -> dict:
    score = 0.85 if expected.lower()[:6] in question.lower() else 0.4
    return {"q": question, "expected": expected, "score": score, "hallucination": score < 0.3}

print(validate_silver_layer())
print(audit_mlflow_experiment())
print(evaluate_chatbot_response("¿Cuál es la temperatura?", "valor numérico"))


{'asset_id': 'AULA01', 'tags_present': True, 'completeness_pct': 99.2, 'outliers': 0, 'verdict': 'OK'}
{'experiment': 'case_B_baseline_2026', 'n_runs': 4, 'best_MAE': 12.4, 'lakefs_tag_referenced': True, 'verdict': 'OK'}
{'q': '¿Cuál es la temperatura?', 'expected': 'valor numérico', 'score': 0.4, 'hallucination': False}


## 10. Exploración paso a paso

Combinamos en un mini-agente.


In [3]:
TOOLS = {
    "validate_silver_layer": validate_silver_layer,
    "audit_mlflow_experiment": audit_mlflow_experiment,
    "evaluate_chatbot_response": evaluate_chatbot_response,
}

def call_agent(plan: list[tuple[str, dict]]) -> list[dict]:
    out = []
    for name, kwargs in plan:
        out.append({"tool": name, "result": TOOLS[name](**kwargs)})
    return out

plan = [
    ("validate_silver_layer", {"asset_id": "AULA01"}),
    ("audit_mlflow_experiment", {"name": "case_B_baseline_2026"}),
    ("evaluate_chatbot_response", {"question": "¿Cuál fue la T media ayer?", "expected": "valor numérico"}),
]
results = call_agent(plan)
results


[{'tool': 'validate_silver_layer',
  'result': {'asset_id': 'AULA01',
   'tags_present': True,
   'completeness_pct': 99.2,
   'outliers': 0,
   'verdict': 'OK'}},
 {'tool': 'audit_mlflow_experiment',
  'result': {'experiment': 'case_B_baseline_2026',
   'n_runs': 4,
   'best_MAE': 12.4,
   'lakefs_tag_referenced': True,
   'verdict': 'OK'}},
 {'tool': 'evaluate_chatbot_response',
  'result': {'q': '¿Cuál fue la T media ayer?',
   'expected': 'valor numérico',
   'score': 0.4,
   'hallucination': False}}]

## 11. Transformación bronce → plata

No aplica.


## 12. Construcción de capa oro

Informe consolidado.


In [4]:
report = pd.DataFrame([{**{"tool": r["tool"]}, **r["result"]} for r in results])
report


,tool,asset_id,tags_present,completeness_pct,outliers,verdict,experiment,n_runs,best_MAE,lakefs_tag_referenced,q,expected,score,hallucination
0,validate_silver_layer,AULA01,True,99.2,0.0,OK,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,audit_mlflow_experiment,NaN,NaN,NaN,NaN,OK,case_B_baseline_2026,4.0,12.4,True,NaN,NaN,NaN,NaN
2,evaluate_chatbot_response,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,¿Cuál fue la T media ayer?,valor numérico,0.4,False


## 13. Visualizaciones explicativas

Verdict por tool.


In [5]:
report["tool"].value_counts().plot.bar(color="#3F51B5")
plt.title("Tools invocadas")
plt.tight_layout()


## 14. Validaciones

Cada tool retorna un dict válido y JSON-serializable.


In [6]:
import json
for r in results:
    json.dumps(r["result"])
print("Serialización OK")


Serialización OK


## 15. Errores comunes

1. Tools que devuelven None.
2. Captura silenciosa de excepciones — el agente no detecta el fallo.
3. No registrar timing por tool.


## 16. Ejercicios propuestos

1. Reemplaza el evaluador de chatbot por embeddings simples.
2. Añade `audit_data_drift(period)`.
3. Conecta a un LLM real con `OPENAI_API_KEY`.


## 17. Cómo se reutiliza con datos reales

Las tools no cambian: solo el origen de los datos.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `08_case_H_rag_chatbot/01_arquitectura_rag_tools.ipynb`.
- Documento web del caso: `docs/use-cases/case-g-data-quality-agents.md`.


## 19. Marco teórico (nivel doctoral)

### Reglas de calidad jerárquicas

Sea $\mathcal{D}_b$ bronce, $\mathcal{D}_s$ plata, $\mathcal{D}_o$ oro.
Score por capa:

$$
\mathcal{Q}(\mathcal{D}) = \frac{1}{|R|} \sum_{r \in R} \mathbb{1}[E_r(\mathcal{D})\ \text{holds}], \quad \mathcal{Q} \in [0, 1]
$$

| Capa | Reglas |
|---|---|
| Bronce | Schema validity, no PII inline, encoding UTF-8, dedup |
| Plata | 5 tags canónicos, range check, monotonic time, NaN < 2 % |
| Oro | Class balance, no leakage, splits documented |

### Drift detection — KL divergence

$$
D_{KL}(P \parallel Q) = \sum_x P(x) \log \frac{P(x)}{Q(x)}
$$

aplicado entre histogramas $P$ (training) y $Q$ (production). Alerta si
$D_{KL} > 0.1$.

### Agentes especialistas (LLM con tools)

$$
\text{Agent}_i = \langle \pi_i, \mathcal{T}_i, \mathcal{M}_i \rangle
$$

con $\pi_i$ política (prompt), $\mathcal{T}_i$ toolkit, $\mathcal{M}_i$
memoria. Composición vía pipeline:

$$
\text{Output} = \pi_n(\pi_{n-1}(\cdots \pi_1(\text{Input})))
$$


## 20. Visión corporativa CAPTIA

### Propuesta de valor

Calidad de datos es **transversal**: sin ella ningún caso de uso tiene valor. Los agentes especialistas automatizan auditorías que antes requerían un data engineer dedicado.

### ROI estimado

| Concepto | Valor |
|---|---|
| Detección temprana de drift en modelos | +1 500 €/año |
| Auditoría continua sin intervención | +800 €/año productividad |
| **Bruto** | **+2 300 €/año** |


## 21. Bibliografía y referencias

- Schelter, S. et al. (2018). *Automating Large-Scale Data Quality Verification*. VLDB.
- Great Expectations. *Documentation*. https://greatexpectations.io
- Anthropic (2024). *Claude API — Tools*. https://docs.anthropic.com
- Polyzotis, N. et al. (2017). *Data Lifecycle Challenges in Production Machine Learning*. SIGMOD.
